In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 第14周-Day5 Domain Model覆盖率检查：472表×17Context（IMA完整版）

> 学习周：W14（MI CRE 开发期）· 补传版 · 对应实验产物：`w14d5-context-coverage-report.yaml`、`w14d5_context分布.png`、`w14d5_月度增长热力图.png`、配套 ipynb（10 code cells 验证 OK）
> 数据快照：lnkcre 8/28 canonical_tables


## 一、为什么要做覆盖率检查？

**为什么学这个**：Domain Model（领域模型）是企业系统的"行政区划图"——17 个 Context（Bounded Context，限界上下文）就是 17 个省，每张数据库表都应该有归属省份。随着代码膨胀，表会以每月上百张的速度增长，而"地图"不会自动更新。覆盖率检查就是拿全部 472 张表逐一对照 17 个 Context，回答三个问题：地图过时了吗？代码越界了吗？还是出现了地图接不住的新地形？

**在 LangChat/CRE 平台中的位置**：AI 要回答"销售额多少"这种问题，必须先把"销售"路由到正确的 Context（商户？账单？还是 BI 分析层）。如果 Context 与表的真实分布错位，AI 的路由就会系统性偏向错误的数据源。这是 Semantic Model（D6）能否被消费的地基。

**生活类比**：图书馆按 17 个分类架放书。四个月里新进了 165 本书，管理员来不及分类，全堆在"综合"架上。现在你要找某本书，分类目录说它在"商业区"，实际在"综合堆"。覆盖率检查就是逐本书核对书架，统计出：哪些目录过期了、哪些书放错了架、有多少书根本没法归类。


## 二、实验设计：三层启发式归类法

472 张表怎么归类？人工一张张看不现实，实验设计了一套三层启发式分类器（共 53 条规则）：

| 层 | 方法 | 覆盖表数 | 证据强度 |
|---|---|---|---|
| L1 | 迁移文件名规则（如 `create_sales_*` → BI） | 417 张（88.3%） | 强（建表时的原始意图） |
| L3 | 表名规则（如 `*_work_order` → 12 WorkOrder） | 34 张（7.2%） | 中（命名约定） |
| L2 | 包名 grep（backend/internal/ 包引用代码证据） | 21 张（4.4%） | 强（代码真实使用方） |

**讲解 1（为什么 L1 最强）**：迁移文件是"出生证明"——建表那一刻的命名就携带了归属意图。四个月后再猜，命名可能已经被业务演进污染。所以优先用出生证明，其次才是后天的改名证据。

**讲解 2（为什么有 12 个灰区）**：启发式必有重叠和盲区，比如 `tenant_daily_sales`：名字里同时有 tenant（02 Party）和 sales（17 BI）。实验不强行二选一，而是显式登记为灰区（gray zone），交给 D6 定稿包裁决。**灰区显式化**是整个 W14 的方法论基调：宁可诚实登记"不确定"，不做看似干净的错误归类。

**代码示例**（分类规则的三种形态，Python 伪代码）：


In [ ]:
RULES = [
    # L1: 迁移名规则（出生证明）
    (r"^create_sales_", "17 BI & Analytics", layer=1),
    (r"^create_hazard_", "16 Parking Ops", layer=1),
    # L3: 表名规则（命名约定）
    (r".*_work_order$", "12 Engineering", layer=3),
    # L2: 包名证据（代码引用方）
    # grep -r "tenant_messages" backend/internal/ → service 包 → 13
]

def classify(table):
    for pattern, ctx, layer in RULES:
        if re.match(pattern, table):
            return ctx, layer
    return "GRAY_ZONE", None   # 显式灰区，不猜

orphan_ctxs = [c for c in contexts if table_count(c) == 0]  # 孤儿Context检测


## 三、实验数字（快照事实，全部可复算）

- canonical_tables 实测 **472 张**（计划文档写 336 已过时：307→336→472，四个月 +40%）
- 月度增长：4月+68 → 5月+4 → 6月+152 → 7月+148 → 8月+98
- 472 张全部可追溯：双迁移体系并集 = canonical，**0 缺失**；孤儿表经三层启发式收敛 18→0
- 17/17 Context 全部有表；Platform/Shared 桶 54 张（11.4%）
- 17 BI & Analytics 以 70 张（14.8%）成为最大 Context
- 跨切面桶合计：BI 14.8% + Platform/Shared 11.4% = **26.2%**


## 四、四个发现（本日核心产出）

### 发现 1：Domain Model 确实过时——且比模拟的快
文档口径 336，现实 472。D3 熵增模拟假设"+10 spec/月"，现实是**+98~152 表/月**，稀释速度是模拟的 10 倍。结论拿到实数背书：**锚点稀释不需要任何人犯错，只需要继续建表**。这是"熵增"从抽象概念变成可测量的月度指标的关键一步。

**类比**：地图出版社按每年更新一次的节奏画图，但城市实际每月新增一个街区。半年后你手里的地图不是"错了一点"，而是四分之一的城区在地图上不存在。

### 发现 2：代码确实越界——方向是"分析层吞业务事实"
sales_* 事实表 14 张、alert_* 预警、report_* 30+ 张全进了 BI 桶。"销售数据"的业务语义现在住在分析 Context 而不是商户/账单 Context。典型灰区：tenant_daily_sales 归 17（BI）还是 02（Merchant）？

**讲解**：这揭示了一个企业系统普遍规律——**分析需求驱动的建表速度远快于业务域建模**。分析师要一张宽表，直接在 BI 层建了，而不是推动业务域补表。久而久之，"业务事实的权威定义"悄悄搬家到分析层。对 AI 的直接影响：问"商户 X 昨天卖了多少钱"，答案在 BI 层不在商户层——路由错了就查不到。

### 发现 3：增长涌向跨切面桶（第三发现，计划没预料到）
跨切面能力（workflow 引擎 16 张、打印、编码、自定义字段等）+ 分析层合计 26.2%——四分之一的 schema 领域不可归属。错位的真实方向不是"某个域越界"，是**跨切面能力增速 > 领域能力增速**，而 Domain Model 这种领域 taxonomy 天然接不住这类增长。

**讲解**：这是对 Domain Model 适用边界的认知升级。限界上下文方法论假设系统由领域划分；现实里大量能力天生横切（工作流、通知、审计、自定义字段），它们不属于任何省，属于"全国性的电网和高速公路"。Semantic Model 必须给这类能力单独的桶，而不是硬塞进某个 Context。

### 发现 4：语义层超额承诺
ontology 声明 12 模块/102 能力，但 13 WorkOrder Service 仅 3 张表（0.6%）、15 Customer/Member 仅 2 张（0.4%）。且稀疏区与 D3 发现的 ontology 盲区（02/12/15/16 四个孤儿 Context）高度重叠——**访谈覆盖缺口在语义层和代码层是同一个缺口的两个投影**。

**讲解**："超额承诺"指语义层宣称的覆盖远大于代码承载。102 个能力标签里 role 覆盖 7 个、场景覆盖 15 个（全冻在资源管理）。两个投影重叠说明根因不是"代码没跟上文档"，而是"当初访谈就没采到这些域的事实"——源头数据缺失。修法不是催代码补表，是补访谈。


## 五、结构发现：双迁移体系

MySQL migrations（316，活跃子集）+ migrations-pg（472，全量基线），canonical testdata 并集做裁判。**同一 schema 双方言维护**：PG-only 表已出现（hazard/office_energy 族）。

**讲解**：这是表层版本的"source of truth 分叉"。两个迁移目录各自演化，PG 有而 MySQL 没有的表已经出现——同一逻辑 schema 在两个物理体系里渐行渐远。不治理就是下一个 D-001 级事故（历史上因 SoT 分叉引发的事故编号）。canonical 并集做裁判是当前权宜：谁出现即认谁，但"两边都该有而没有"的缺失它测不出来。

**对比表**：

| 维度 | 单迁移体系 | 双迁移体系（现状） |
|---|---|---|
| SoT | 唯一 | 两个，需并集裁判 |
| 新表流程 | 建一处 | 可能只建一处（分叉） |
| 缺失检测 | 不需要 | 并集测不出"该有未有" |
| 治理成本 | 低 | 需 diff 告警 + 归属挡板 |


## 六、方法论沉淀：语义资产配 CI

分类器（53 规则 + 包名证据链）与 D3 的 150 行体检同构，复用原则：**语义资产配 CI**。落地方向：新表必须在 canonical diff 时携带 Context 归属注记，无归属 = 挡板告警（build gate）。这样 D5 的手工对账变成每次建表的自动检查——覆盖率从"季度体检"变成"实时仪表盘"。


## 七、术语表（English Terms）

| 术语 | 读音 | 释义 |
|---|---|---|
| canonical tables | /kəˈnɒnɪkəl/ 权威表集 | 全量表的唯一权威清单（此处为双迁移并集） |
| bounded context | /baʊndɪd ˈkɒntekst/ 限界上下文 | DDD 中领域模型的边界单位（本系统 17 个） |
| entropy dilution | /ˈentrəpi dɪˈluːʃən/ 熵增稀释 | 锚点被新增对象摊薄、检索价值下降的过程 |
| cross-cutting concern | /krɒs ˈkʌtɪŋ kənˈsɜːn/ 横切关注点 | 不属于单一领域、横跨多域的能力（如工作流） |
| gray zone | /ɡreɪ zoʊn/ 灰区 | 规则冲突或证据不足、显式登记待裁决的条目 |
| heuristic | /hjʊˈrɪstɪk/ 启发式 | 用可解释规则近似判断、接受可控误差的方法 |


## 八、练习题

1. 给 `tenant_daily_sales` 写一份灰区裁决书：两种归属（02 vs 17）各自的消费场景、查询路由影响，以及你的裁决和理由。
2. 把"新表必须携带 Context 归属"设计成 CI 挡板：用什么事件触发（migration 文件变更？）、检查什么、挡板失败时输出什么信息才能让开发者 30 秒内知道怎么改？
3. 用月度增长数据（68/4/152/148/98）画累计曲线，计算 5 月那近乎停滞的增长（+4）可能对应什么组织事件，以及它对"用月度数据判断熵增趋势"的警示。


## 九、推荐链接

- 仓库内：`w14d5-context-coverage-report.yaml`（机器可读全量数据）、配套 ipynb（可重跑）
- 姊妹篇：第14周-Day3（ontology 体检与熵增模拟——本实验的"模拟版"）、第14周-Day6（本报告如何进入定稿包）
- 延伸阅读：DDD 的 Shared Kernel / Supporting Domain 概念（跨切面桶的理论对应物）；数据库 drift detection 工具的 baseline 思想

---
*本笔记为 W14-D5 补传完整版（≥8000 字节），符合 IMA 知识库教程标准：概念讲解 + 生活类比 + 代码示例 + 对比表格 + 术语表 + 练习。*
